In [1]:
!nvidia-smi

Wed Aug 12 16:15:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd ~
!git clone https://github.com/sahilahmed21/Atlas.git Atlas || true
%cd ~/Atlas
# Phase 7 live harness requires tip >= 517a309 (not older origin tips).
!git fetch origin
!git checkout master
!git pull --ff-only origin master
!git log -1 --oneline
!uv run python benchmarks/run_routing_matrix.py --help | grep -E "worker-mode|patterns" || echo "STOP: old tip — no --worker-mode (will silently run simulated Phase 5)"

/root
fatal: destination path 'Atlas' already exists and is not an empty directory.
/root/Atlas


In [4]:
!pip install -U pip
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 52.9 MB/s  0:00:00


In [3]:
!uv sync

Resolved 111 packages in 1ms
Checked 108 packages in 1ms


In [4]:
!uv run python --version

Python 3.11.15


In [7]:
!pip install "https://github.com/vllm-project/vllm/releases/download/v0.26.0/vllm-0.26.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 MB 23.2 MB/s  0:00:16
INFO: pip is looking at multiple versions of quack-kernels to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 15.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 52.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 85.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 85.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.7/767.7 kB 38.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 MB 60.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 97.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 76.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 54.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.8/755.8 kB 40.8 MB/s  0:00:00
   ━━━━━━━━━━━

In [5]:
!python scripts/verify_wsl_vllm.py

vllm 0.26.0 (pin ok) 0.26.0
torch 2.11.0+cu128
cuda True
device Tesla T4


In [9]:
!python -c "import vllm; print(vllm.__version__)"

0.26.0


In [6]:
%cd ~/Atlas

!CUDA_VISIBLE_DEVICES=0 nohup uv run vllm serve Qwen/Qwen2.5-0.5B-Instruct \
  --port 8001 \
  --gpu-memory-utilization 0.4 \
  --max-model-len 2048 \
  > /tmp/vllm-8001.log 2>&1 &

/root/Atlas


In [10]:
!cat /tmp/vllm-8001.log

(APIServer pid=4897) INFO 08-12 16:16:16 [api_utils.py:345] 
(APIServer pid=4897) INFO 08-12 16:16:16 [api_utils.py:345]        █     █     █▄   ▄█
(APIServer pid=4897) INFO 08-12 16:16:16 [api_utils.py:345]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.26.0
(APIServer pid=4897) INFO 08-12 16:16:16 [api_utils.py:345]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=4897) INFO 08-12 16:16:16 [api_utils.py:345]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=4897) INFO 08-12 16:16:16 [api_utils.py:345] 
(APIServer pid=4897) INFO 08-12 16:16:16 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'port': 8001, 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'max_model_len': 2048, 'gpu_memory_utilization': 0.4}
(APIServer pid=4897) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=4897) INFO 08-12 16:16:33 [model.py:623] Resolved architecture: Qwen2ForCau

In [14]:
!curl -s http://127.0.0.1:8001/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-0.5B-Instruct","object":"model","created":1786551580,"owned_by":"vllm","root":"Qwen/Qwen2.5-0.5B-Instruct","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-a01a2e3490f75b75","object":"model_permission","created":1786551580,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [8]:
%cd ~/Atlas

!CUDA_VISIBLE_DEVICES=0 nohup uv run vllm serve Qwen/Qwen2.5-0.5B-Instruct \
  --port 8002 \
  --gpu-memory-utilization 0.4 \
  --max-model-len 2048 \
  > /tmp/vllm-8002.log 2>&1 &

/root/Atlas


In [12]:
!cat /tmp/vllm-8002.log

(APIServer pid=5063) INFO 08-12 16:16:58 [api_utils.py:345] 
(APIServer pid=5063) INFO 08-12 16:16:58 [api_utils.py:345]        █     █     █▄   ▄█
(APIServer pid=5063) INFO 08-12 16:16:58 [api_utils.py:345]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.26.0
(APIServer pid=5063) INFO 08-12 16:16:58 [api_utils.py:345]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=5063) INFO 08-12 16:16:58 [api_utils.py:345]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=5063) INFO 08-12 16:16:58 [api_utils.py:345] 
(APIServer pid=5063) INFO 08-12 16:16:58 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'port': 8002, 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'max_model_len': 2048, 'gpu_memory_utilization': 0.4}
(APIServer pid=5063) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=5063) INFO 08-12 16:16:58 [model.py:623] Resolved architecture: Qwen2ForCau

In [15]:
!curl -s http://127.0.0.1:8002/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-0.5B-Instruct","object":"model","created":1786551589,"owned_by":"vllm","root":"Qwen/Qwen2.5-0.5B-Instruct","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-99d2539fc1af6b72","object":"model_permission","created":1786551589,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [18]:
import requests

payload = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [{"role": "user", "content": "Say ping"}],
    "max_tokens": 10,
    "stream": False,
}

r = requests.post(
    "http://127.0.0.1:8001/v1/chat/completions",
    json=payload,
    timeout=60,
)

print(r.status_code)
print(r.text)

200
{"id":"chatcmpl-8f14af58125db1f8","object":"chat.completion","created":1786551640,"model":"Qwen/Qwen2.5-0.5B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"Hello! How can I assist you today?","refusal":null,"annotations":null,"audio":null,"function_call":null,"reasoning":null},"logprobs":null,"finish_reason":"stop","stop_reason":null,"token_ids":null,"routed_experts":null}],"service_tier":null,"system_fingerprint":"vllm-0.26.0-4f1bb46d","usage":{"prompt_tokens":31,"total_tokens":41,"completion_tokens":10,"prompt_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"prompt_text":null,"kv_transfer_params":null,"ec_transfer_params":null,"metrics":null}


In [19]:
import requests

payload = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [{"role": "user", "content": "Say ping"}],
    "max_tokens": 10,
    "stream": False,
}

r = requests.post(
    "http://127.0.0.1:8002/v1/chat/completions",
    json=payload,
    timeout=60,
)

print(r.status_code)
print(r.text)

200
{"id":"chatcmpl-a4c5ed000cdf08cc","object":"chat.completion","created":1786551666,"model":"Qwen/Qwen2.5-0.5B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"Hello! How can I assist you today?","refusal":null,"annotations":null,"audio":null,"function_call":null,"reasoning":null},"logprobs":null,"finish_reason":"stop","stop_reason":null,"token_ids":null,"routed_experts":null}],"service_tier":null,"system_fingerprint":"vllm-0.26.0-4f1bb46d","usage":{"prompt_tokens":31,"total_tokens":41,"completion_tokens":10,"prompt_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"prompt_text":null,"kv_transfer_params":null,"ec_transfer_params":null,"metrics":null}


## Phase 7 live matrix — gate before run

**2026-08-12 session:** dual vLLM on :8001/:8002 **PASS**. Matrix cells below wrote 
esults/phase5/routing_matrix.csv with worker_mode=simulated because Colab was on tip **before** 517a309 (CLI flags ignored). That CSV is **not** Phase 7 evidence — see docs/phases/phase-7/RUN_LOG.md.

**Before re-running:** tip >= 517a309, and the next cell must print --worker-mode. If it prints STOP, do not run the matrix.


In [ ]:
%cd ~/Atlas
!git log -1 --oneline
!uv run python benchmarks/run_routing_matrix.py --help | grep -E "worker-mode|patterns" || (echo "STOP: old tip — no --worker-mode" && exit 1)
!test -f results/phase5-live/README.md && echo "phase5-live landing zone OK"

In [20]:
# Valid live run: must print "wrote .../results/phase5-live/routing_matrix_live.csv"
# and rows with worker_mode=live. If it writes results/phase5/ + simulated — STOP (old tip).
%cd ~/Atlas
!uv run python benchmarks/run_routing_matrix.py \
  --worker-mode live \
  --patterns high_reuse \
  --strategies round_robin,prefix_aware \
  --n 24 \
  --hardware colab-t4 \
  --vllm-version 0.26.0 \
  --replica-mode time_sliced_dual \
  --worker-a-url http://127.0.0.1:8001/v1 \
  --worker-b-url http://127.0.0.1:8002/v1
!ls -lh results/phase5-live/
!python -c "import pandas as pd; p='results/phase5-live/routing_matrix_live.csv'; df=pd.read_csv(p); assert (df['worker_mode']=='live').all(), list(df['worker_mode'].unique()); print(df.to_string(index=False))"

/root/Atlas


In [21]:
# DEPRECATED duplicate — use cell above. Kept only for history; do not run.
print("Skip: use the gated live matrix cell above (writes results/phase5-live/).")

/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
wrote /root/Atlas/results/phase5/routing_matrix.csv (12 rows)
high_reuse   round_robin   p50=147.5 p95=281.25 hit%=0.0 skew=0.5
high_reuse   least_load    p50=297.5 p95=556.25 hit%=0.0 skew=1.0
high_reuse   prefix_aware  p50=297.5 p95=556.25 hit%=95.83 skew=1.0
low_reuse    round_robin   p50=237.5 p95=371.25 hit%=0.0 skew=0.5
low_reuse    least_load    p50=387.5 p95=646.25 hit%=0.0 skew=1.0
low_reuse    prefix_aware  p50=387.5 p95=646.25 hit%=0.0 skew=1.0
bursty       round_robin   p50=167.5 p95=281.25 hit%=0.0 skew=0.5
bursty       least_load    p50=297.5 p95=556.25 hit%=0.0 skew=1.0
bursty       prefix_aware  p50=297.5 p95=556.25 hit%=91.67 skew=1.0
steady       round_robin   p50=147.5 p95=281.25 hit%=0.0 skew=0.5
steady       least_loa

In [22]:
!ls -lh results/phase5-live/

total 4.0K
-rw-r--r-- 1 root root 351 Aug 12 16:07 README.md


In [32]:
# DEPRECATED duplicate matrix — do not run (2026-08-12 accidentally produced sim CSV).
print("Skip: duplicate of live matrix; use gated cell that asserts worker_mode=live.")

/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
wrote /root/Atlas/results/phase5/routing_matrix.csv (12 rows)
high_reuse   round_robin   p50=147.5 p95=281.25 hit%=0.0 skew=0.5
high_reuse   least_load    p50=297.5 p95=556.25 hit%=0.0 skew=1.0
high_reuse   prefix_aware  p50=297.5 p95=556.25 hit%=95.83 skew=1.0
low_reuse    round_robin   p50=237.5 p95=371.25 hit%=0.0 skew=0.5
low_reuse    least_load    p50=387.5 p95=646.25 hit%=0.0 skew=1.0
low_reuse    prefix_aware  p50=387.5 p95=646.25 hit%=0.0 skew=1.0
bursty       round_robin   p50=167.5 p95=281.25 hit%=0.0 skew=0.5
bursty       least_load    p50=297.5 p95=556.25 hit%=0.0 skew=1.0
bursty       prefix_aware  p50=297.5 p95=556.25 hit%=91.67 skew=1.0
steady       round_robin   p50=147.5 p95=281.25 hit%=0.0 skew=0.5
steady       least_loa

In [33]:
!ls -lah /root/Atlas/results/phase5-live

total 12K
drwxr-xr-x 2 root root 4.0K Aug 12 16:07 .
drwxr-xr-x 7 root root 4.0K Aug 12 16:07 ..
-rw-r--r-- 1 root root  351 Aug 12 16:07 README.md


In [35]:
# Prefer live CSV. phase5/routing_matrix.csv from 2026-08-12 Colab is simulated-only.
import pandas as pd
from pathlib import Path

live = Path("results/phase5-live/routing_matrix_live.csv")
sim = Path("results/phase5/routing_matrix.csv")
path = live if live.is_file() else sim
df = pd.read_csv(path)
print("loaded:", path)
print("worker_mode unique:", df["worker_mode"].unique().tolist())
if path == sim or (df["worker_mode"] == "simulated").any():
    print("WARNING: simulated rows — NOT Phase 7 GPU evidence")
display(df)

,pattern,strategy,n,ttft_p50_ms,ttft_p95_ms,tokens_per_s_mean,cache_hit_pct,worker_skew,worker_counts,worker_mode
0,high_reuse,round_robin,24,147.5,281.25,8.685,0.00,0.5,"{'worker-a': 12, 'worker-b': 12}",simulated
1,high_reuse,least_load,24,297.5,556.25,5.491,0.00,1.0,{'worker-a': 24},simulated
2,high_reuse,prefix_aware,24,297.5,556.25,5.491,95.83,1.0,{'worker-a': 24},simulated
3,low_reuse,round_robin,24,237.5,371.25,4.808,0.00,0.5,"{'worker-a': 12, 'worker-b': 12}",simulated
4,low_reuse,least_load,24,387.5,646.25,3.350,0.00,1.0,{'worker-a': 24},simulated
5,low_reuse,prefix_aware,24,387.5,646.25,3.350,0.00,1.0,{'worker-a': 24},simulated
6,bursty,round_robin,24,167.5,281.25,8.222,0.00,0.5,"{'worker-a': 12, 'worker-b': 12}",simulated
7,bursty,least_load,24,297.5,556.25,5.401,0.00,1.0,{'worker-a': 24},simulated
8,bursty,prefix_aware,24,297.5,556.25,5.401,91.67,1.0,{'worker-a': 24},simulated
9,steady,round_robin,24,147.5,281.25,8.685,0.00,0.5,"{'worker-a': 12, 'worker-b': 12}",simulated


In [36]:
print(df.columns.tolist())
print(df.to_string(index=False))

['pattern', 'strategy', 'n', 'ttft_p50_ms', 'ttft_p95_ms', 'tokens_per_s_mean', 'cache_hit_pct', 'worker_skew', 'worker_counts', 'worker_mode']
   pattern     strategy  n  ttft_p50_ms  ttft_p95_ms  tokens_per_s_mean  cache_hit_pct  worker_skew                    worker_counts worker_mode
high_reuse  round_robin 24        147.5       281.25              8.685           0.00          0.5 {'worker-a': 12, 'worker-b': 12}   simulated
high_reuse   least_load 24        297.5       556.25              5.491           0.00          1.0                 {'worker-a': 24}   simulated
high_reuse prefix_aware 24        297.5       556.25              5.491          95.83          1.0                 {'worker-a': 24}   simulated
 low_reuse  round_robin 24        237.5       371.25              4.808           0.00          0.5 {'worker-a': 12, 'worker-b': 12}   simulated
 low_reuse   least_load 24        387.5       646.25              3.350           0.00          1.0                 {'worker-a': 24

In [37]:
print("Rows:", len(df))
print("Patterns:", df["pattern"].unique())
print("Strategies:", df["strategy"].unique())

Rows: 12
Patterns: ['high_reuse' 'low_reuse' 'bursty' 'steady']
Strategies: ['round_robin' 'least_load' 'prefix_aware']


In [38]:
!uv run python benchmarks/run_routing_matrix.py --help

/root/Atlas/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
wrote /root/Atlas/results/phase5/routing_matrix.csv (12 rows)
high_reuse   round_robin   p50=147.5 p95=281.25 hit%=0.0 skew=0.5
high_reuse   least_load    p50=297.5 p95=556.25 hit%=0.0 skew=1.0
high_reuse   prefix_aware  p50=297.5 p95=556.25 hit%=95.83 skew=1.0
low_reuse    round_robin   p50=237.5 p95=371.25 hit%=0.0 skew=0.5
low_reuse    least_load    p50=387.5 p95=646.25 hit%=0.0 skew=1.0
low_reuse    prefix_aware  p50=387.5 p95=646.25 hit%=0.0 skew=1.0
bursty       round_robin   p50=167.5 p95=281.25 hit%=0.0 skew=0.5
bursty       least_load    p50=297.5 p95=556.25 hit%=0.0 skew=1.0
bursty       prefix_aware  p50=297.5 p95=556.25 hit%=91.67 skew=1.0
steady       round_robin   p50=147.5 p95=281.25 hit%=0.0 skew=0.5
steady       least_loa

In [41]:
!grep -n "worker-mode\|patterns\|strategies\|phase5-live\|routing_matrix" benchmarks/run_routing_matrix.py

1:"""Replay frozen traffic traces across routing strategies; write Phase 5 CSV."""
28:DEFAULT_OUT = ROOT / "results" / "phase5" / "routing_matrix.csv"
154:    patterns: list[str] | None = None,
155:    strategies: list[str] | None = None,
162:    patterns = list(patterns or PATTERNS)
163:    strategies = list(strategies or STRATEGIES)
169:        for pattern in patterns:
172:            for strategy in strategies:
